# 10.1 Markov decision processes: practice notebook

Plan when actions have chance outcomes. Describe the problem as states, actions, transitions and rewards, then find the best policy with the Bellman equation, value iteration, policy iteration and Q-values.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Value iteration on the racing car

In [ ]:
# Value iteration on the racing car MDP (Berkeley CS188). The car can be cool or warm. Overheated ends the race.
# T[state][action] is a list of (probability, next state, reward).
T = {"cool": {"slow": [(1.0, "cool", 1)],
              "fast": [(0.5, "cool", 2), (0.5, "warm", 2)]},
     "warm": {"slow": [(0.5, "cool", 1), (0.5, "warm", 1)],
              "fast": [(1.0, "over", -10)]}}
gamma = 0.9
V = {"cool": 0.0, "warm": 0.0, "over": 0.0}

for k in range(1, 4):
    new = {"over": 0.0}
    for s in T:
        best = None
        for a in T[s]:
            q = sum(p * (r + gamma * V[s2]) for p, s2, r in T[s][a])
            if best is None or q > best:
                best = q
        new[s] = best
    V = new
    print("after sweep", k, ":", {s: round(V[s], 3) for s in ["cool", "warm"]})

### Policy iteration on the racing car

In [ ]:
# Policy iteration on the same racing car MDP. Policy evaluation solves two linear equations exactly.
T = {"cool": {"slow": [(1.0, "cool", 1)],
              "fast": [(0.5, "cool", 2), (0.5, "warm", 2)]},
     "warm": {"slow": [(0.5, "cool", 1), (0.5, "warm", 1)],
              "fast": [(1.0, "over", -10)]}}
g = 0.9

def evaluate(pi):
    # V(s) = sum p (r + g V(s')). Write it as a V(cool) + b V(warm) = c for each state, then solve.
    rows = []
    for s in ["cool", "warm"]:
        a_c, a_w, c = 0.0, 0.0, 0.0
        for p, s2, r in T[s][pi[s]]:
            c = c + p * r
            if s2 == "cool":
                a_c = a_c + g * p
            if s2 == "warm":
                a_w = a_w + g * p
        rows.append((a_c, a_w, c))
    (p1, q1, c1), (p2, q2, c2) = rows
    # (1 - p1) Vc - q1 Vw = c1 and -p2 Vc + (1 - q2) Vw = c2
    det = (1 - p1) * (1 - q2) - q1 * p2
    vc = (c1 * (1 - q2) + q1 * c2) / det
    vw = ((1 - p1) * c2 + p2 * c1) / det
    return {"cool": vc, "warm": vw, "over": 0.0}

pi = {"cool": "slow", "warm": "slow"}
while True:
    V = evaluate(pi)
    print("policy", pi, "-> V =", {s: round(V[s], 2) for s in ["cool", "warm"]})
    new = {}
    for s in T:
        new[s] = max(T[s], key=lambda a: sum(p * (r + g * V[s2]) for p, s2, r in T[s][a]))
    if new == pi:
        break
    pi = new
print("optimal policy:", pi)

## Your turn

### Task 1: Discounted return

Write `discounted_return(rewards, gamma)` = Σ gammaᵗ × rewards[t].

Hint: Keep a factor that starts at 1 and is multiplied by gamma after each reward.

In [ ]:
def discounted_return(rewards, gamma):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('discounted_return([1, 1, 1], 0.5)', 1.75), ('round(discounted_return([-0.04] * 7 + [1], 1), 2)', 0.72), ('round(discounted_return([0, 0, 10], 0.9), 2)', 8.1), ('discounted_return([], 0.9)', 0.0)])

### Task 2: One Bellman update

Write `bellman(V, gamma)` for the racing car T: return a new dictionary with V(s) = max over a of Σ p (r + gamma V(s′)) for "cool" and "warm", and 0.0 for "over".

Hint: For each state, compute a Q value per action and keep the largest.

In [ ]:
T = {"cool": {"slow": [(1.0, "cool", 1)],
              "fast": [(0.5, "cool", 2), (0.5, "warm", 2)]},
     "warm": {"slow": [(0.5, "cool", 1), (0.5, "warm", 1)],
              "fast": [(1.0, "over", -10)]}}   # T[s][a] = list of (probability, next state, reward)

def bellman(V, gamma):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("bellman({'cool': 0.0, 'warm': 0.0, 'over': 0.0}, 0.9)", {'cool': 2.0, 'warm': 1.0, 'over': 0.0}), ("{s: round(v, 2) for s, v in bellman({'cool': 2.0, 'warm': 1.0, 'over': 0.0}, 0.9).items()}", {'cool': 3.35, 'warm': 2.35, 'over': 0.0}), ("{s: round(v, 2) for s, v in bellman({'cool': 15.5, 'warm': 14.5, 'over': 0.0}, 0.9).items()}", {'cool': 15.5, 'warm': 14.5, 'over': 0.0})])

### Task 3: Value iteration with the policy

Write `solve(gamma)` for the racing car: repeat Bellman updates from all zeros until the largest change is below 1e-9, then return (V, policy), where policy maps "cool" and "warm" to the best action.

Hint: Loop until the change is tiny, then pick the action with the largest Q value in each state.

In [ ]:
T = {"cool": {"slow": [(1.0, "cool", 1)],
              "fast": [(0.5, "cool", 2), (0.5, "warm", 2)]},
     "warm": {"slow": [(0.5, "cool", 1), (0.5, "warm", 1)],
              "fast": [(1.0, "over", -10)]}}   # T[s][a] = list of (probability, next state, reward)

def solve(gamma):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('solve(0.9)[1]', {'cool': 'fast', 'warm': 'slow'}), ("round(solve(0.9)[0]['cool'], 2)", 15.5), ('solve(0.1)[1]', {'cool': 'fast', 'warm': 'slow'}), ("round(solve(0.5)[0]['warm'], 3)", 2.5)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Discounted return

Test 3: 10 × 0.9² = 8.1.

In [ ]:
def discounted_return(rewards, gamma):
    total = 0.0
    factor = 1.0
    for r in rewards:
        total = total + factor * r
        factor = factor * gamma
    return total

In [ ]:
check([('discounted_return([1, 1, 1], 0.5)', 1.75), ('round(discounted_return([-0.04] * 7 + [1], 1), 2)', 0.72), ('round(discounted_return([0, 0, 10], 0.9), 2)', 8.1), ('discounted_return([], 0.9)', 0.0)])

### Solution to task 2: One Bellman update

Test 3: the optimal values do not change under the update, since they solve the Bellman equation.

In [ ]:
T = {"cool": {"slow": [(1.0, "cool", 1)],
              "fast": [(0.5, "cool", 2), (0.5, "warm", 2)]},
     "warm": {"slow": [(0.5, "cool", 1), (0.5, "warm", 1)],
              "fast": [(1.0, "over", -10)]}}   # T[s][a] = list of (probability, next state, reward)

def bellman(V, gamma):
    new = {"over": 0.0}
    for s in T:
        best = None
        for a in T[s]:
            q = 0.0
            for p, s2, r in T[s][a]:
                q = q + p * (r + gamma * V[s2])
            if best is None or q > best:
                best = q
        new[s] = best
    return new

In [ ]:
check([("bellman({'cool': 0.0, 'warm': 0.0, 'over': 0.0}, 0.9)", {'cool': 2.0, 'warm': 1.0, 'over': 0.0}), ("{s: round(v, 2) for s, v in bellman({'cool': 2.0, 'warm': 1.0, 'over': 0.0}, 0.9).items()}", {'cool': 3.35, 'warm': 2.35, 'over': 0.0}), ("{s: round(v, 2) for s, v in bellman({'cool': 15.5, 'warm': 14.5, 'over': 0.0}, 0.9).items()}", {'cool': 15.5, 'warm': 14.5, 'over': 0.0})])

### Solution to task 3: Value iteration with the policy

Test 4, γ = 0.5: V(cool) − V(warm) = 1 still holds, and V(warm) = 1.25 + 0.5 V(warm) gives V(warm) = 2.5.

In [ ]:
T = {"cool": {"slow": [(1.0, "cool", 1)],
              "fast": [(0.5, "cool", 2), (0.5, "warm", 2)]},
     "warm": {"slow": [(0.5, "cool", 1), (0.5, "warm", 1)],
              "fast": [(1.0, "over", -10)]}}   # T[s][a] = list of (probability, next state, reward)

def q_value(V, s, a, gamma):
    total = 0.0
    for p, s2, r in T[s][a]:
        total = total + p * (r + gamma * V[s2])
    return total

def solve(gamma):
    V = {"cool": 0.0, "warm": 0.0, "over": 0.0}
    while True:
        new = {"over": 0.0}
        change = 0.0
        for s in T:
            best = max(q_value(V, s, a, gamma) for a in T[s])
            new[s] = best
            change = max(change, abs(best - V[s]))
        V = new
        if change < 1e-9:
            break
    policy = {}
    for s in T:
        best_a = None
        for a in T[s]:
            if best_a is None or q_value(V, s, a, gamma) > q_value(V, s, best_a, gamma):
                best_a = a
        policy[s] = best_a
    return V, policy

In [ ]:
check([('solve(0.9)[1]', {'cool': 'fast', 'warm': 'slow'}), ("round(solve(0.9)[0]['cool'], 2)", 15.5), ('solve(0.1)[1]', {'cool': 'fast', 'warm': 'slow'}), ("round(solve(0.5)[0]['warm'], 3)", 2.5)])